# Paper Figures — All from Cached Data

Generates all publication figures from Drive-cached finegrain results.
No GPU needed.

In [ ]:
import numpy as np
import json, os
from scipy import stats, optimize
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['figure.dpi'] = 150

from google.colab import drive
drive.mount('/content/drive')

R = Path('/content/drive/MyDrive/LRTIA/Results')
FIG_DIR = Path('/content/drive/MyDrive/LRTIA/Paper_Figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONTEXT = 100
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for c in curves:
        interp = np.interp(common_x, np.array(c['ctx_lengths']), np.array(c['ppls']),
                           left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def get_corrected_marginal(intact, shuffled):
    ip = compute_raw_ppl_curve(intact)
    sp = compute_raw_ppl_curve(shuffled)
    return -np.diff(ip) - (-np.diff(sp))

def get_binned(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    return np.array(bc), np.array(bm)

def fit_power_law(marg):
    bc, bm = get_binned(marg)
    if len(bc) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return {'slope': slope, 'r': r, 'p': p, 'bc': bc, 'bm': bm, 'intercept': intercept}
    return None

def load_pair(ip, sp):
    with open(ip) as f: intact = json.load(f)
    with open(sp) as f: shuffled = json.load(f)
    return intact, shuffled

print('Setup done')

In [ ]:
# === Load ALL cached results ===
DATA = {}  # key -> {intact, shuffled, marg, fit, label, ...}

def add_dataset(key, label, ip, sp, category='human'):
    if not (ip.exists() and sp.exists()): return
    intact, shuffled = load_pair(ip, sp)
    if len(intact) < 5 or len(shuffled) < 5: return
    marg = get_corrected_marginal(intact, shuffled)
    fit = fit_power_law(marg)
    if fit is None: return
    DATA[key] = {'intact': intact, 'shuffled': shuffled, 'marg': marg,
                 'fit': fit, 'label': label, 'n': len(intact), 'category': category}

# Llama Wiki
for code, name in [('zh','Chinese'), ('ja','Japanese'), ('ko','Korean'),
                    ('tr','Turkish'), ('ar','Arabic'), ('fi','Finnish')]:
    add_dataset(f'llama_{code}_wiki', f'{name} Wiki',
                R/'Llama_crosslingual'/f'wiki_{code}_intact.json',
                R/'Llama_crosslingual'/f'wiki_{code}_shuffled.json', 'human_wiki')

# Llama Spoken
add_dataset('llama_buckeye', 'Buckeye spoken',
            R/'Llama_crosslingual'/'buckeye_intact.json',
            R/'Llama_crosslingual'/'buckeye_shuffled.json', 'human_spoken')
add_dataset('llama_french', 'French spoken',
            R/'Llama_crosslingual'/'french_intact.json',
            R/'Llama_crosslingual'/'french_shuffled.json', 'human_spoken')

# Llama English Wiki + AI
add_dataset('llama_en_wiki', 'English Wiki',
            R/'AI_vs_Human_wiki'/'llama_en_human_intact.json',
            R/'AI_vs_Human_wiki'/'llama_en_human_shuffled.json', 'human_wiki_en')
add_dataset('llama_en_ai', 'English AI',
            R/'AI_vs_Human_wiki'/'llama_en_ai_intact.json',
            R/'AI_vs_Human_wiki'/'llama_en_ai_shuffled.json', 'ai')
add_dataset('llama_zh_ai', 'Chinese AI',
            R/'AI_vs_Human_wiki'/'llama_zh_ai_intact.json',
            R/'AI_vs_Human_wiki'/'llama_zh_ai_shuffled.json', 'ai')
add_dataset('llama_tr_ai', 'Turkish AI',
            R/'AI_vs_Human_wiki'/'llama_tr_ai_intact.json',
            R/'AI_vs_Human_wiki'/'llama_tr_ai_shuffled.json', 'ai')

# Mistral Wiki
for code, name in [('zh','Chinese'), ('ja','Japanese'), ('ko','Korean'),
                    ('tr','Turkish'), ('ar','Arabic'), ('fi','Finnish')]:
    add_dataset(f'mistral_{code}_wiki', f'{name} Wiki (M)',
                R/'Wiki_multilingual_finegrain'/f'{code}_intact_v1.json',
                R/'Wiki_multilingual_finegrain'/f'{code}_shuffled_v1.json', 'human_wiki_mistral')

# Mistral Spoken
add_dataset('mistral_buckeye', 'Buckeye spoken (M)',
            R/'Buckeye_finegrain'/'buckeye_intact_v1.json',
            R/'Buckeye_finegrain'/'buckeye_shuffled_v1.json', 'human_spoken_mistral')
add_dataset('mistral_french', 'French spoken (M)',
            R/'French_oral_finegrain'/'french_oral_intact_v1.json',
            R/'French_oral_finegrain'/'french_oral_shuffled_v1.json', 'human_spoken_mistral')

# Mistral AI
add_dataset('mistral_en_wiki', 'English Wiki (M)',
            R/'AI_vs_Human_wiki'/'mistral_en_human_intact.json',
            R/'AI_vs_Human_wiki'/'mistral_en_human_shuffled.json', 'human_wiki_en_mistral')
add_dataset('mistral_en_ai', 'English AI (M)',
            R/'AI_vs_Human_wiki'/'mistral_en_ai_intact.json',
            R/'AI_vs_Human_wiki'/'mistral_en_ai_shuffled.json', 'ai_mistral')
add_dataset('mistral_zh_ai', 'Chinese AI (M)',
            R/'AI_vs_Human_wiki'/'mistral_zh_ai_intact.json',
            R/'AI_vs_Human_wiki'/'mistral_zh_ai_shuffled.json', 'ai_mistral')
add_dataset('mistral_tr_ai', 'Turkish AI (M)',
            R/'AI_vs_Human_wiki'/'mistral_tr_ai_intact.json',
            R/'AI_vs_Human_wiki'/'mistral_tr_ai_shuffled.json', 'ai_mistral')

# News (Llama)
for code, name in [('en','English'), ('fr','French'), ('tr','Turkish'), ('zh','Chinese')]:
    add_dataset(f'llama_{code}_news', f'{name} News',
                R/'News_finegrain'/f'llama_{code}_news_intact.json',
                R/'News_finegrain'/f'llama_{code}_news_shuffled.json', 'human_news')

print(f'Loaded {len(DATA)} datasets')
for k, v in DATA.items():
    print(f'  {v["label"]:<25} α={v["fit"]["slope"]:>7.3f}  r={v["fit"]["r"]:>7.3f}  n={v["n"]}  [{v["category"]}]')

In [ ]:
# === FIGURE 1: Power Law Decay Curves — Human vs AI ===
# Show actual corrected marginals overlaid for 3 languages (Llama)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (lang, name) in enumerate([('zh', 'Chinese'), ('tr', 'Turkish'), ('en', 'English')]):
    ax = axes[idx]
    hkey = f'llama_{lang}_wiki'
    akey = f'llama_{lang}_ai'
    if hkey not in DATA or akey not in DATA: continue
    h, a = DATA[hkey], DATA[akey]

    # Smoothed corrected marginals
    ax.plot(common_x[1:], uniform_filter1d(h['marg'], 5), '-', color='#2196F3',
            linewidth=2, label=f'Human (α={h["fit"]["slope"]:.2f})')
    ax.plot(common_x[1:], uniform_filter1d(a['marg'], 5), '-', color='#f44336',
            linewidth=2, label=f'AI (α={a["fit"]["slope"]:.2f})')

    # Power law fit lines
    x_fit = np.linspace(2, 90, 200)
    for d, color, ls in [(h, '#2196F3', '--'), (a, '#f44336', '--')]:
        y_fit = np.exp(d['fit']['intercept']) * x_fit ** d['fit']['slope']
        ax.plot(x_fit, y_fit, color=color, linestyle=ls, alpha=0.4, linewidth=1)

    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_xlabel('Context Distance (tokens)')
    if idx == 0: ax.set_ylabel('Corrected Marginal\n(coherence signal)')
    ax.set_title(f'{name} Wikipedia', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)
    ax.set_xlim(1, 100)

plt.suptitle('Figure 1: Coherence Decay — Human vs AI-Generated Text (Llama-3-8B probe)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_human_vs_ai_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig1')

In [ ]:
# === FIGURE 2: Functional Form Comparison ===
# Fit 5 models to representative datasets, show which wins

def power_law_fn(x, a, b): return a * np.power(x, -b)
def exponential_fn(x, a, b): return a * np.exp(-b * x)
def stretched_fn(x, a, b, c): return a * np.exp(-b * np.power(x, c))
def log_fn(x, a, b): return a - b * np.log(x)
def linear_fn(x, a, b): return a - b * x

FUNC_MODELS = {
    'Power law': {'func': power_law_fn, 'p0': [1,0.75], 'k': 2, 'color': '#1f77b4'},
    'Exponential': {'func': exponential_fn, 'p0': [1,0.05], 'k': 2, 'color': '#d62728'},
    'Stretched exp': {'func': stretched_fn, 'p0': [1,0.05,0.5], 'k': 3, 'color': '#9467bd'},
    'Logarithmic': {'func': log_fn, 'p0': [1,0.1], 'k': 2, 'color': '#2ca02c'},
    'Linear': {'func': linear_fn, 'p0': [1,0.01], 'k': 2, 'color': '#8c564b'},
}

# Pick 4 representative datasets
show = ['llama_zh_wiki', 'llama_tr_wiki', 'llama_buckeye', 'llama_french']
show = [s for s in show if s in DATA]

fig, axes = plt.subplots(1, len(show), figsize=(5*len(show), 4.5))
if len(show) == 1: axes = [axes]

for idx, key in enumerate(show):
    ax = axes[idx]
    d = DATA[key]
    bc, bm = d['fit']['bc'], d['fit']['bm']
    ax.plot(bc, bm, 'ko', markersize=7, zorder=10, label='Data')
    x_sm = np.linspace(bc.min(), bc.max(), 200)

    fits = {}
    for fname, spec in FUNC_MODELS.items():
        try:
            bounds = (0, np.inf) if fname != 'Logarithmic' else (-np.inf, np.inf)
            popt, _ = optimize.curve_fit(spec['func'], bc, bm, p0=spec['p0'],
                                         maxfev=10000, bounds=bounds)
            y_pred = spec['func'](bc, *popt)
            rss = np.sum((bm - y_pred)**2)
            tss = np.sum((bm - np.mean(bm))**2)
            r2 = 1 - rss/tss if tss > 0 else 0
            aic = len(bc) * np.log(rss/len(bc)) + 2*spec['k'] if rss > 0 else np.inf
            fits[fname] = {'popt': popt, 'r2': r2, 'aic': aic}
        except: pass

    best = min(fits, key=lambda k: fits[k]['aic']) if fits else None
    for fname, spec in FUNC_MODELS.items():
        if fname not in fits: continue
        is_best = fname == best
        y_sm = spec['func'](x_sm, *fits[fname]['popt'])
        lbl = f"{fname} (R²={fits[fname]['r2']:.3f})"
        if is_best: lbl += ' ★'
        ax.plot(x_sm, y_sm, color=spec['color'],
                linestyle='-' if is_best else '--',
                linewidth=2.5 if is_best else 1,
                alpha=1.0 if is_best else 0.4, label=lbl)

    ax.set_xscale('log')
    ax.set_xlabel('Distance (tokens)')
    if idx == 0: ax.set_ylabel('Corrected Marginal')
    ax.set_title(d['label'], fontweight='bold')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.15)

plt.suptitle('Figure 2: Functional Form Comparison — Power Law vs Alternatives',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_functional_form.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig2')

In [ ]:
# === FIGURE 3: Cross-Linguistic Convergence — Both Probes ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Llama exponents
ax = axes[0]
llama_human = [(k, v) for k, v in DATA.items()
               if k.startswith('llama_') and v['category'] in ('human_wiki', 'human_spoken')
               and 'en' not in k]
llama_human.sort(key=lambda x: x[1]['fit']['slope'])
labels = [v['label'] for _, v in llama_human]
exps = [v['fit']['slope'] for _, v in llama_human]
colors = ['#2196F3' if 'Wiki' in l else '#4CAF50' for l in labels]

y_pos = range(len(labels))
ax.barh(y_pos, exps, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Power Law Exponent (α)')
ax.set_title('Llama-3-8B', fontweight='bold', fontsize=12)
mean_l = np.mean(exps)
ax.axvline(mean_l, color='black', linewidth=2, alpha=0.5, label=f'Mean: {mean_l:.2f}')
ax.axvline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler (1991)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15, axis='x')
for i, exp in enumerate(exps):
    ax.text(exp - 0.02, i, f'{exp:.2f}', va='center', ha='right', fontsize=8, fontweight='bold')

# Panel B: Mistral exponents
ax = axes[1]
mistral_human = [(k, v) for k, v in DATA.items()
                 if k.startswith('mistral_') and v['category'] in ('human_wiki_mistral', 'human_spoken_mistral')]
mistral_human.sort(key=lambda x: x[1]['fit']['slope'])
labels_m = [v['label'].replace(' (M)', '') for _, v in mistral_human]
exps_m = [v['fit']['slope'] for _, v in mistral_human]
colors_m = ['#ff7f0e' if 'Wiki' in l else '#e377c2' for l in labels_m]

y_pos = range(len(labels_m))
ax.barh(y_pos, exps_m, color=colors_m, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_m, fontsize=10)
ax.set_xlabel('Power Law Exponent (α)')
ax.set_title('Mistral-7B', fontweight='bold', fontsize=12)
mean_m = np.mean(exps_m)
ax.axvline(mean_m, color='black', linewidth=2, alpha=0.5, label=f'Mean: {mean_m:.2f}')
ax.axvline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler (1991)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15, axis='x')
for i, exp in enumerate(exps_m):
    ax.text(exp - 0.02, i, f'{exp:.2f}', va='center', ha='right', fontsize=8, fontweight='bold')

plt.suptitle('Figure 3: Cross-Linguistic Convergence — Two Independent Probes\n(non-English human text only)',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_crosslingual_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Llama: mean={mean_l:.3f}, SD={np.std(exps):.3f}')
print(f'Mistral: mean={mean_m:.3f}, SD={np.std(exps_m):.3f}')
print('Saved fig3')

In [ ]:
# === FIGURE 4: Per-Speaker Distribution (Buckeye) ===

if 'llama_buckeye' in DATA:
    d = DATA['llama_buckeye']
    # Get per-speaker fits
    speaker_ids = sorted(set(c['doc_id'] for c in d['intact']))
    speaker_exps = []
    for sid in speaker_ids:
        si = [c for c in d['intact'] if c['doc_id'] == sid]
        ss = [c for c in d['shuffled'] if c['doc_id'] == sid]
        if len(si) < 2 or len(ss) < 2: continue
        marg = get_corrected_marginal(si, ss)
        fit = fit_power_law(marg)
        if fit and fit['r'] < -0.5:
            speaker_exps.append(fit['slope'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.hist(speaker_exps, bins=12, color='#4CAF50', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(speaker_exps), color='darkgreen', linewidth=2,
               label=f'Mean: {np.mean(speaker_exps):.2f}')
    ax.axvline(-0.77, color='gray', linestyle=':', linewidth=2,
               label='Anderson & Schooler')
    ax.set_xlabel('Decay Exponent (α)')
    ax.set_ylabel('Count')
    ax.set_title('Per-Speaker Exponent Distribution', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)

    ax = axes[1]
    sorted_exps = sorted(speaker_exps)
    ax.bar(range(len(sorted_exps)), sorted_exps, color='#4CAF50', alpha=0.6)
    ax.axhline(np.mean(speaker_exps), color='darkgreen', linewidth=2)
    ax.axhline(-0.77, color='gray', linestyle=':', linewidth=2, label='Anderson & Schooler')
    ax.set_xlabel('Speaker (sorted)')
    ax.set_ylabel('Decay Exponent (α)')
    ax.set_title('Individual Speakers (sorted)', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)

    plt.suptitle('Figure 4: Per-Speaker Coherence Decay — Buckeye Spoken English (Llama-3-8B)',
                 fontsize=13, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig4_per_speaker.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Per-speaker: mean={np.mean(speaker_exps):.3f}, SD={np.std(speaker_exps):.3f}, n={len(speaker_exps)}')
    print('Saved fig4')
else:
    print('No Buckeye data')

In [ ]:
# === FIGURE 5: Absolute Magnitude — Human vs AI at Different Distances ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Absolute corrected marginals at short/mid/long for all Llama datasets
ax = axes[0]

categories = [
    ('Non-English\nHuman Wiki', [k for k in DATA if k.startswith('llama_') and
                                  DATA[k]['category'] == 'human_wiki'], '#2196F3'),
    ('Spoken', [k for k in DATA if k.startswith('llama_') and
                DATA[k]['category'] == 'human_spoken'], '#4CAF50'),
    ('English\nWiki', ['llama_en_wiki'] if 'llama_en_wiki' in DATA else [], '#ff9800'),
    ('AI', [k for k in DATA if k.startswith('llama_') and 'ai' in DATA[k]['category']], '#f44336'),
]

x_pos = 0
for cat_name, keys, color in categories:
    if not keys: continue
    long_vals = []
    for k in keys:
        bc, bm = DATA[k]['fit']['bc'], DATA[k]['fit']['bm']
        long_val = bm[-2] if len(bm) > 2 else bm[-1]
        long_vals.append(long_val)
    mean_long = np.mean(long_vals)
    sem_long = np.std(long_vals) / np.sqrt(len(long_vals)) if len(long_vals) > 1 else 0
    ax.bar(x_pos, mean_long, yerr=sem_long, capsize=5,
           color=color, alpha=0.7, edgecolor='black', width=0.7)
    ax.text(x_pos, mean_long + sem_long + 0.005, f'{mean_long:.3f}',
            ha='center', fontsize=9, fontweight='bold')
    x_pos += 1

ax.set_xticks(range(x_pos))
ax.set_xticklabels([c[0] for c in categories if c[1]], fontsize=10)
ax.set_ylabel('Mean Corrected Marginal\n(long range: 30-50 tokens)')
ax.set_title('Long-Range Coherence Signal', fontweight='bold')
ax.grid(True, alpha=0.15, axis='y')

# Panel B: Full decay curves for human Chinese vs AI Chinese vs English Wiki
ax = axes[1]
for key, color, ls in [('llama_zh_wiki', '#2196F3', '-'),
                        ('llama_zh_ai', '#f44336', '-'),
                        ('llama_en_wiki', '#ff9800', '--')]:
    if key not in DATA: continue
    d = DATA[key]
    bc, bm = d['fit']['bc'], d['fit']['bm']
    ax.plot(bc, bm, 'o-', color=color, linewidth=2, markersize=5,
            label=f'{d["label"]} (α={d["fit"]["slope"]:.2f})')

ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens)')
ax.set_ylabel('Corrected Marginal')
ax.set_title('Binned Decay: Human vs AI vs English', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

plt.suptitle('Figure 5: Absolute Coherence Magnitude — Long-Range Signal Comparison',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_absolute_magnitude.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig5')

In [ ]:
# === FIGURE 6: Grand Summary — All Exponents in One Plot ===

fig, ax = plt.subplots(figsize=(14, 8))

# Organize by category
groups = [
    ('Non-English Wiki\n(Llama)', [(k, v) for k, v in DATA.items()
        if v['category'] == 'human_wiki' and k.startswith('llama_')], '#2196F3'),
    ('Non-English Wiki\n(Mistral)', [(k, v) for k, v in DATA.items()
        if v['category'] == 'human_wiki_mistral'], '#ff7f0e'),
    ('Spoken\n(Llama)', [(k, v) for k, v in DATA.items()
        if v['category'] == 'human_spoken'], '#4CAF50'),
    ('Spoken\n(Mistral)', [(k, v) for k, v in DATA.items()
        if v['category'] == 'human_spoken_mistral'], '#8BC34A'),
    ('News\n(Llama)', [(k, v) for k, v in DATA.items()
        if v['category'] == 'human_news'], '#9C27B0'),
    ('English Wiki', [(k, v) for k, v in DATA.items()
        if v['category'] in ('human_wiki_en', 'human_wiki_en_mistral')], '#ff9800'),
    ('AI', [(k, v) for k, v in DATA.items()
        if 'ai' in v['category']], '#f44336'),
]

y_pos = 0
y_ticks, y_labels = [], []
group_positions = []

for group_name, items, color in groups:
    if not items: continue
    items.sort(key=lambda x: x[1]['fit']['slope'])
    group_start = y_pos
    for key, v in items:
        exp = v['fit']['slope']
        ax.barh(y_pos, exp, color=color, alpha=0.7, edgecolor='black', linewidth=0.5, height=0.7)
        ax.text(exp - 0.02, y_pos, f'{exp:.2f}', va='center', ha='right', fontsize=7, fontweight='bold')
        y_ticks.append(y_pos)
        label = v['label'].replace(' (M)', ' [M]')
        y_labels.append(label)
        y_pos += 1
    group_positions.append((group_start, y_pos - 1, group_name, color))
    y_pos += 0.5  # gap between groups

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel('Power Law Exponent (α)', fontsize=12)
ax.axvline(-0.77, color='gray', linestyle=':', linewidth=1.5, alpha=0.7, label='Anderson & Schooler -0.77')
ax.axvline(-0.78, color='black', linestyle='-', linewidth=2, alpha=0.3, label='Non-English human mean -0.78')
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, alpha=0.15, axis='x')
ax.set_title('Figure 6: Complete Landscape of Coherence Decay Exponents', fontweight='bold', fontsize=13)

# Group labels on right
for gs, ge, gname, gcolor in group_positions:
    mid = (gs + ge) / 2
    ax.text(0.02, mid, gname, transform=ax.get_yaxis_transform(),
            fontsize=8, fontweight='bold', color=gcolor, va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=gcolor, alpha=0.8))

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_grand_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig6')

In [ ]:
# === Summary stats for paper text ===
print('='*70)
print('STATS FOR PAPER')
print('='*70)

# Non-English human wiki (Llama)
ne_llama = [v['fit']['slope'] for k, v in DATA.items() if v['category'] == 'human_wiki']
print(f'\nNon-English Wiki (Llama): mean={np.mean(ne_llama):.3f}, SD={np.std(ne_llama):.3f}, n={len(ne_llama)}')

# Non-English human wiki (Mistral)
ne_mistral = [v['fit']['slope'] for k, v in DATA.items() if v['category'] == 'human_wiki_mistral']
print(f'Non-English Wiki (Mistral): mean={np.mean(ne_mistral):.3f}, SD={np.std(ne_mistral):.3f}, n={len(ne_mistral)}')

# Spoken (Llama)
sp_llama = [v['fit']['slope'] for k, v in DATA.items() if v['category'] == 'human_spoken']
print(f'Spoken (Llama): mean={np.mean(sp_llama):.3f}, SD={np.std(sp_llama):.3f}, n={len(sp_llama)}')

# AI (Llama)
ai_llama = [v['fit']['slope'] for k, v in DATA.items() if v['category'] == 'ai']
print(f'AI (Llama): mean={np.mean(ai_llama):.3f}, SD={np.std(ai_llama):.3f}, n={len(ai_llama)}')

# All human non-English (both probes)
all_human_ne = ne_llama + ne_mistral + sp_llama
print(f'\nAll non-English human (both probes): mean={np.mean(all_human_ne):.3f}, SD={np.std(all_human_ne):.3f}, n={len(all_human_ne)}')

# English written
en_written = [v['fit']['slope'] for k, v in DATA.items()
              if v['category'] in ('human_wiki_en', 'human_wiki_en_mistral')]
if en_written:
    print(f'English written: mean={np.mean(en_written):.3f}, SD={np.std(en_written):.3f}')

print(f'\nAnderson & Schooler (1991): -0.77')
print(f'Kahana & Adler (2002) range: -0.64 to -1.03')